### Understanding the Features We Are Extracting
Instead of guessing what is inside the tumor, we use classic Computer Vision math to capture three distinct things:

GLCM (Gray-Level Co-occurrence Matrix): This analyzes how often pairs of pixels with specific intensities happen next to each other. It calculates statistical metrics like Contrast (how sharp the intensity changes are), Energy (texture uniformity), and Homogeneity (how smooth the tumor tissue looks).

LBP (Local Binary Patterns): This looks at every single pixel, compares it to its immediate neighbors, and flags whether it is an edge, a flat surface, or a corner point. It is excellent for micro-texture classification in medical scans.

Shape Features: We will calculate the physical geometry of the original tumor using its bounding box width, height, and aspect ratio ($\text{Aspect Ratio} = \frac{\text{width}}{\text{height}}$). Malignant thyroid nodules often grow vertically ("taller-than-wide"), making shape metrics highly predictive.

### Step 3.1: Install the Radiomics Toolkit

In [1]:
%pip install scikit-image

   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.9 MB 2.8 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/11.9 MB 2.3 MB/s eta 0:00:05
   ------ --------------------------------- 1.8/11.9 MB 2.5 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/11.9 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.9/11.9 MB 2.5 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/11.9 MB 2.4 MB/s eta 0:00:04
   ------------ --------------------------- 3.7/11.9 MB 2.3 MB/s eta 0:00:04
   -------------- ------------------------- 4.2/11.9 MB 2.3 MB/s eta 0:00:04
   --------------- ------------------------ 4.7/11.9 MB 2.3 MB/s eta 0:00:04
   ---------------- ----------------------- 5.0/11.9 MB 2.3 MB/s eta 0:00:04
   ----------------- ---------------------- 5.2/11.9 MB 2.2 MB/s eta 0:00:04
   ----------

### Radiomics Feature Extractor

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

# 1. Path Definitions
cropped_dir = "../Outputs/cropped_images"
mapping_csv_path = "../Outputs/image_label_mapping.csv"
output_csv_path = "../Outputs/radiomics_features.csv"

# 2. Load our master mapping spreadsheet to get labels and shape bounds
mapping_df = pd.read_csv(mapping_csv_path)

# This list will hold the final radiomics vectors for all 5,000 files
radiomics_data = []

print("Extracting Radiomics (GLCM + LBP + Shape) for 5,000 nodules...")

for idx, row in mapping_df.iterrows():
    img_name = row['image_name']
    img_path = os.path.join(cropped_dir, img_name)
    
    # Read image in Grayscale mode (Texture analysis requires a single channel)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
        
    # --- FEATURE SET 1: GEOMETRIC SHAPE FEATURES ---
    # Derived from the original unpadded bounding box dimensions
    width = float(row['xmax'] - row['xmin'])
    height = float(row['ymax'] - row['ymin'])
    aspect_ratio = width / height if height != 0 else 1.0
    bounding_box_area = width * height
    
    # --- FEATURE SET 2: GLCM TEXTURE FEATURES ---
    # Compute the matrix for a distance of 1 pixel in 4 primary directions (0, 45, 90, 135 degrees)
    glcm = graycomatrix(img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                        levels=256, symmetric=True, normed=True)
    
    # Extract mean properties across all directions to keep features robust
    contrast = np.mean(graycoprops(glcm, 'contrast'))
    correlation = np.mean(graycoprops(glcm, 'correlation'))
    energy = np.mean(graycoprops(glcm, 'energy'))
    homogeneity = np.mean(graycoprops(glcm, 'homogeneity'))
    
    # --- FEATURE SET 3: LOCAL BINARY PATTERNS (LBP) ---
    # Extract structural micro-patterns (8 surrounding pixels at radius 1)
    lbp = local_binary_pattern(img, P=8, R=1, method='uniform')
    
    # Convert the LBP texture image map into a normalized 10-bin histogram vector
    # (Uniform LBP with 8 points yields exactly 10 distinct binary patterns)
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.linspace(0, 11, 11), density=True)
    
    # --- COMBINE INTO SINGLE ROW VECTOR ---
    feature_row = {
        "image_name": img_name,
        "label": row['label'],
        "shape_width": width,
        "shape_height": height,
        "shape_aspect_ratio": aspect_ratio,
        "shape_area": bounding_box_area,
        "glcm_contrast": contrast,
        "glcm_correlation": correlation,
        "glcm_energy": energy,
        "glcm_homogeneity": homogeneity
    }
    
    # Append the 10 separate LBP bins dynamically into our dictionary
    for b in range(10):
        feature_row[f"lbp_bin_{b}"] = lbp_hist[b]
        
    radiomics_data.append(feature_row)
    
    # Progress indicator
    if (idx + 1) % 1000 == 0:
        print(f" Successfully vectorized {idx + 1} / 5000 textures...")

# 3. Save everything to a structured dataset file
features_df = pd.DataFrame(radiomics_data)
features_df.to_csv(output_csv_path, index=False)

print(f"\nSuccess! Radiomics feature matrix saved to: {output_csv_path}")
print(f"Matrix Dimension: {features_df.shape[0]} samples x {features_df.shape[1]} columns.")

Extracting Radiomics (GLCM + LBP + Shape) for 5,000 nodules...
 Successfully vectorized 1000 / 5000 textures...
 Successfully vectorized 2000 / 5000 textures...
 Successfully vectorized 3000 / 5000 textures...
 Successfully vectorized 4000 / 5000 textures...
 Successfully vectorized 5000 / 5000 textures...

Success! Radiomics feature matrix saved to: ../Outputs/radiomics_features.csv
Matrix Dimension: 5000 samples x 20 columns.


### Verify

In [2]:
import pandas as pd
df_features = pd.read_csv("../Outputs/radiomics_features.csv")
df_features.head()

,image_name,label,shape_width,shape_height,shape_aspect_ratio,shape_area,glcm_contrast,glcm_correlation,glcm_energy,glcm_homogeneity,lbp_bin_0,lbp_bin_1,lbp_bin_2,lbp_bin_3,lbp_bin_4,lbp_bin_5,lbp_bin_6,lbp_bin_7,lbp_bin_8,lbp_bin_9
0,000001.jpg,0,63.0,41.0,1.536585,2583.0,23.898322,0.991502,0.045255,0.491018,0.012610,0.005272,0.095700,0.273365,0.302191,0.074864,0.049498,0.079230,0.016361,0.0
1,000002.jpg,0,51.0,40.0,1.275000,2040.0,33.644147,0.992533,0.031940,0.419714,0.011922,0.006704,0.085952,0.333046,0.298875,0.057271,0.036635,0.064283,0.014404,0.0
2,000003.jpg,0,162.0,123.0,1.317073,19926.0,15.172731,0.985142,0.034850,0.324308,0.030275,0.022503,0.128384,0.325563,0.219899,0.074085,0.033319,0.038773,0.036290,0.0
3,000004.jpg,0,110.0,70.0,1.571429,7700.0,81.064601,0.972290,0.038203,0.353678,0.027213,0.019622,0.119869,0.313804,0.245282,0.073396,0.036073,0.042233,0.031598,0.0
4,000005.jpg,0,103.0,87.0,1.183908,8961.0,10.381040,0.995491,0.040894,0.397375,0.019205,0.011813,0.098145,0.337557,0.265013,0.071204,0.033228,0.049390,0.023535,0.0


In [1]:
%pip install torch torchvision

   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---